## Robustness of networks using MILP and Fast-Lin

### Analyzes how the robustness of networks depends on variables such as dense vs CNN, number of layers, and width of layers.

Let $f: [0, 1]^{n_0} \to \mathbb{R}^{10}$ denote a neural network trained to classify numbers 0-9, where the network classifies image $x$ being labelled as $i$ more likely than $j$ if $f_i(x) > f_j(x)$.

Let the image $x_0$ be classified as $i$ by the network. We find non-trivial $\epsilon$ such that $f_i(x) > f_j(x)$ for all $x \in B_p(x_0, \epsilon)$ and $i \neq j$.

In [ ]:
from matplotlib import pyplot as plt
import torch
from torchvision import datasets

from architectures.networkArchitectures import networkRegistry
from robustness.exactRobustness import exactRobustness
from robustness.fastLin import fastLin
from training.networkTraining import transform
from utils.extractParams import extractParams
from utils.loadNetwork import loadNetwork

In [ ]:
testingSet = datasets.MNIST(root = "./data", train = False, transform = transform, download = True)

### Computes certified lower bounds

Gets certified lower bounds by Fast-Lin. Could be fast enough for all considered networks even unoptimized.

In [ ]:
certifiedEpsilonsPerNetwork = []
for networkName, networkEntry in networkRegistry:
    network = loadNetwork(networkName)

    parameters = extractParams(network = network, inputShape = (1, 28, 28))
    weights = [W for W, _ in parameters]
    biases = [b for _, b in parameters]

    certifiedEpsilonsPerImage = []
    for i in range(100):
        image, label = testingSet[i]
    
        pNorm = 1
        x0 = image.view(-1).numpy()

        output = network(image)
        _, predictedClass = torch.max(output, 1)
        predictedClass = predictedClass.item()
        targetClasses = [targetClass for targetClass in range(0, 10) if targetClass != predictedClass]
        
        certifiedEpsilon, _, _ = fastLin(weights = weights, biases = biases, x0 = x0, pNorm = pNorm, epsilon0 = 10, originalClass = predictedClass, targetClasses = targetClasses, tolerance = 0.005)
        certifiedEpsilonsPerImage.append(certifiedEpsilon)

    certifiedEpsilonsPerNetwork.append(certifiedEpsilonsPerImage)

Plot figures comparing robustness vs layer width, number of layers, and total number of neurons.

In [ ]:
# Plot robustness obtained by Fast-Lin

### Computes exact robustness

Get exact robustness by solving multiple MILPs created via the big-M formulation. Infeasible for larger networks.

In [ ]:
epsilonsPerNetwork = []
for networkName, networkEntry in networkRegistry:
    network = loadNetwork(networkName)

    parameters = extractParams(network = network, inputShape = (1, 28, 28))
    weights = [W for W, _ in parameters]
    biases = [b for _, b in parameters]

    epsilonsPerImage = []
    for i in range(100):
        image, label = testingSet[i]
    
        pNorm = 1
        x0 = image.view(-1).numpy()

        output = network(image)
        _, predictedClass = torch.max(output, 1)
        predictedClass = predictedClass.item()
        targetClasses = [targetClass for targetClass in range(0, 10) if targetClass != predictedClass]
        
        epsilon, _, _ = exactRobustness(weights = weights, biases = biases, x0 = x0, pNorm = pNorm, epsilon0 = 10, originalClass = predictedClass, targetClasses = targetClasses)
        epsilonsPerImage.append(epsilon)

    epsilonsPerNetwork.append(epsilonsPerImage)

Plot figures comparing robustness vs layer width, number of layers, and total number of neurons for some smaller networks.

In [ ]:
# Plot robustness obtained by MILP